AES encryption, or Advanced Encryption Standard, is a method for scrambling data to make it unreadable. Imagine you have a secret message and a special lock with a key. AES is like that lock:

* **Data is the message:** The information you want to keep confidential.
* **Encryption is the locking:** AES transforms the data into an unreadable format using a secret key.
* **Decryption is unlocking:** With the same key, you can unscramble the data back to its original form.

AES is widely used because it's secure, efficient, and can be implemented in both hardware and software. It's like a strong padlock that keeps your data safe from prying eyes.


RSA (Rivest-Shamir-Adleman) is a widely used asymmetric cryptographic  algorithm named after its inventors, Ron Rivest, Adi Shamir, and Leonard Adleman. It was first publicly described in 1977. RSA is primarily used for secure data transmission, digital signatures, and key exchange.

### Key Features of RSA

1. **Asymmetric Encryption**:
   RSA uses a pair of keys: a public key and a private key. The public key is used for encryption, while the private key is used for decryption. This means that anyone with the public key can encrypt a message, but only the holder of the private key can decrypt it.

2. **Digital Signatures**:
   RSA can also be used to create digital signatures, which verify the authenticity and integrity of a message. The sender signs a message with their private key, and the receiver can verify the signature using the sender's public key.

3. **Security Basis**:
   The security of RSA is based on the mathematical difficulty of factoring large composite numbers into their prime factors. Specifically, it relies on the fact that while it is easy to multiply two large prime numbers together to form a composite number, it is extremely difficult to reverse the process.

RSA is a fundamental technology in modern cryptography, underpinning many secure communication protocols and systems, including SSL/TLS for secure web browsing, email encryption, and more.

In [4]:
from Crypto.PublicKey import RSA
#security level, 1024-4096 bits
#Generate a key that is 2048 bits long -- industry standard, 8x times faster than 4096, still safe
bits = 2048

#generate  keys
key = RSA.generate(bits)
#get private key & write it to a .pem file
private_key = key.export_key()
file_out = open("private.pem", "wb")
file_out.write(private_key)
file_out.close()

#get public key & write it to a .pem file
public_key = key.publickey().export_key() 
file_out = open("receiver.pem", "wb")
file_out.write(public_key)
file_out.close()

In [6]:
from Crypto.PublicKey import RSA
from Crypto.Random import get_random_bytes
from Crypto.Cipher import AES, PKCS1_OAEP
import binascii 

#sample text to encrypt and send and then decrypt
data = "Nasel sem resitve naslednjega kolokvija.".encode("utf-8")

#we're going write/read binary data, hence "wb"
file_out = open("encrypted_data.txt", "wb")

#get public key
recipient_key = RSA.import_key(open("receiver.pem").read())

#get random AES key: 16, 24 or 32 bytes
session_key = get_random_bytes(16) #this means 128 bit

# Encrypt the session key with the public RSA key
cipher_rsa = PKCS1_OAEP.new(recipient_key) #chiper with padding for extra protection

#encode AES session key
enc_session_key = cipher_rsa.encrypt(session_key)

# Encrypt the data with the AES session key
# EAX is an authenticated encryption with associated data (AEAD) mode 
cipher_aes = AES.new(session_key, AES.MODE_EAX)
#get encrypted text and tags
ciphertext, tag = cipher_aes.encrypt_and_digest(data)
#write all needed to send to the other party: encrypted session key, nonce, tag 
# and encrypted text
#nonce is "number used only once", it helps with extra security
for x in (enc_session_key, cipher_aes.nonce, tag, ciphertext):
    file_out.write(x)
file_out.close()

print("Nonce number (hex):", binascii.hexlify(cipher_aes.nonce).decode('utf-8'))

Nonce number (hex): 2422fa56a4a3f642a16eab61f0019ad0


This Python code snippet demonstrates hybrid encryption, which combines the RSA and AES encryption algorithms to securely encrypt data. Hybrid encryption leverages the strengths of both asymmetric (RSA) and symmetric (AES) encryption methods. Here’s a detailed explanation of each part of the code:

### Importing Required Modules

```python
from Crypto.PublicKey import RSA
from Crypto.Random import get_random_bytes
from Crypto.Cipher import AES, PKCS1_OAEP
```
- `RSA`: Provides functions for generating and handling RSA keys.
- `get_random_bytes`: Generates random bytes for creating secure keys.
- `AES`: Provides functions for AES encryption.
- `PKCS1_OAEP`: A padding scheme for RSA encryption, providing additional security.

### Preparing Data for Encryption

```python
data = "Nasel sem resitve naslednjega kolokvija.".encode("utf-8")
file_out = open("encrypted_data.txt", "wb")
```
- `data`: The plaintext message to be encrypted, encoded in UTF-8.
- `file_out`: Opens a file named `encrypted_data.txt` in binary write mode to store the encrypted data.

### Importing the Recipient's RSA Public Key

```python
recipient_key = RSA.import_key(open("receiver.pem").read())
```
- `recipient_key`: Reads and imports the recipient's RSA public key from a file named `receiver.pem`.

### Generating an AES Session Key

```python
session_key = get_random_bytes(16)
```
- `session_key`: Generates a 16-byte (128-bit) random AES session key. AES keys can be 16, 24, or 32 bytes long (128, 192, or 256 bits, respectively).

### Encrypting the Session Key with RSA

```python
cipher_rsa = PKCS1_OAEP.new(recipient_key)
enc_session_key = cipher_rsa.encrypt(session_key)
```
- `cipher_rsa`: Creates a new PKCS1_OAEP cipher object using the recipient's public key.
- `enc_session_key`: Encrypts the AES session key using the RSA public key. The encrypted session key can only be decrypted by the recipient using their corresponding RSA private key.

### Encrypting the Data with AES

```python
cipher_aes = AES.new(session_key, AES.MODE_EAX)
ciphertext, tag = cipher_aes.encrypt_and_digest(data)
```
- `cipher_aes`: Creates a new AES cipher object in EAX mode using the AES session key. EAX mode provides both encryption and authentication.
- `ciphertext`: The encrypted version of the plaintext `data`.
- `tag`: An authentication tag used to verify the integrity of the encrypted data.

### Writing the Encrypted Data to File

```python
[ file_out.write(x) for x in (enc_session_key, cipher_aes.nonce, tag, ciphertext) ]
file_out.close()
```
- Writes the following components to the output file `encrypted_data.txt`:
  - `enc_session_key`: The RSA-encrypted AES session key.
  - `cipher_aes.nonce`: The nonce (number used once) generated by AES in EAX mode. This nonce is required for decryption.
  - `tag`: The authentication tag generated during AES encryption.
  - `ciphertext`: The AES-encrypted data.
- `file_out.close()`: Closes the file after writing all the encrypted components.

### Summary
This script encrypts a plaintext message using a hybrid encryption approach:
1. It encrypts the plaintext with AES (a symmetric encryption algorithm) using a randomly generated session key.
2. It encrypts the AES session key with RSA (an asymmetric encryption algorithm) using the recipient's public key.
3. It writes the encrypted session key, nonce, authentication tag, and ciphertext to an output file.

The recipient can then:
1. Decrypt the AES session key using their RSA private key.
2. Use the decrypted AES session key, along with the nonce and authentication tag, to decrypt and verify the integrity of the ciphertext.

In [7]:
from Crypto.PublicKey import RSA
from Crypto.Cipher import AES, PKCS1_OAEP

file_in = open("encrypted_data.txt", "rb")

private_key = RSA.import_key(open("private.pem").read())

#private_key.size_in_bytes()  == size of anything encrypted with RSA == fixed size
#16 bytes = normal length of nonce
#16 bytes = normal length of tag
#ciphertext size is the unknown size
enc_session_key, nonce, tag, ciphertext = \
   [ file_in.read(x) for x in (private_key.size_in_bytes(), 16, 16, -1) ]
file_in.close()

# Decrypt the session key with the private RSA key
cipher_rsa = PKCS1_OAEP.new(private_key)
session_key = cipher_rsa.decrypt(enc_session_key)

# Decrypt the data with the AES session key
# N.B.: nonce is now needed at creation of chiper_aes!
cipher_aes = AES.new(session_key, AES.MODE_EAX, nonce)
data = cipher_aes.decrypt_and_verify(ciphertext, tag)
print(data.decode("utf-8"))

Nasel sem resitve naslednjega kolokvija.


This Python script demonstrates the process of decrypting data that was encrypted using a hybrid encryption scheme combining RSA and AES. Below is a detailed step-by-step explanation of each part of the code:

### Importing Necessary Modules

```python
from Crypto.PublicKey import RSA
from Crypto.Cipher import AES, PKCS1_OAEP
```

- `RSA`: Provides functions to handle RSA key generation, import, and operations.
- `AES`: Provides functions for AES encryption and decryption.
- `PKCS1_OAEP`: A padding scheme for RSA encryption that enhances security.

### Opening the Encrypted File

```python
file_in = open("encrypted_data.txt", "rb")
```

- Opens the file `encrypted_data.txt` in binary read mode. This file contains the encrypted session key, nonce, authentication tag, and ciphertext.

### Importing the RSA Private Key

```python
private_key = RSA.import_key(open("private.pem").read())
```

- Reads the RSA private key from a file named `private.pem` and imports it into the `private_key` variable. This private key will be used to decrypt the AES session key.

### Reading Encrypted Components from the File

```python
enc_session_key, nonce, tag, ciphertext = \
   [ file_in.read(x) for x in (private_key.size_in_bytes(), 16, 16, -1) ]
file_in.close()
```

- The file contains several components:
  - `enc_session_key`: The RSA-encrypted AES session key. Its size is determined by the size of the RSA key (in bytes), which is obtained using `private_key.size_in_bytes()`.
  - `nonce`: A 16-byte value used in AES encryption in EAX mode to ensure the uniqueness of the encryption operation.
  - `tag`: A 16-byte authentication tag generated during AES encryption to verify data integrity.
  - `ciphertext`: The actual encrypted data. The size `-1` indicates reading all remaining bytes.

### Decrypting the Session Key

```python
cipher_rsa = PKCS1_OAEP.new(private_key)
session_key = cipher_rsa.decrypt(enc_session_key)
```

- Creates a `PKCS1_OAEP` cipher object using the imported RSA private key (`private_key`).
- Decrypts the `enc_session_key` to retrieve the AES session key (`session_key`). This decrypted session key will be used for AES decryption.

### Decrypting the Data with the AES Session Key

```python
cipher_aes = AES.new(session_key, AES.MODE_EAX, nonce)
data = cipher_aes.decrypt_and_verify(ciphertext, tag)
```

- Creates an AES cipher object (`cipher_aes`) in EAX mode using the decrypted `session_key` and `nonce`.
- Decrypts the `ciphertext` using the AES session key and verifies the integrity of the data using the `tag`.

### Displaying the Decrypted Data

```python
print(data.decode("utf-8"))
```

- Decodes the decrypted data from bytes to a UTF-8 string and prints it.

### Summary

This script performs the following steps:

1. Opens the file containing the encrypted data.
2. Reads the RSA private key.
3. Reads the encrypted session key, nonce, authentication tag, and ciphertext from the file.
4. Decrypts the AES session key using the RSA private key.
5. Decrypts the ciphertext using the AES session key and verifies its integrity.
6. Prints the decrypted plaintext message.

This process ensures that the data is securely decrypted using the private key and the session key, maintaining both confidentiality and integrity of the data.